# ПЗ 6 — Классификация объектов с помощью ResNet

**Задача:** классифицировать кадры через предобученный ResNet50 (ImageNet), получить топ-5 классов для каждого кадра.

**Стек:** `torch`, `torchvision`, `Pillow`, `pandas`

In [ ]:
!pip install torch torchvision Pillow -q

In [ ]:
import os
import json
import torch
import pandas as pd
from PIL import Image
from torchvision import models, transforms
from tqdm.notebook import tqdm

FRAMES_DIR   = '/content/outputs/frames'
OUTPUT_DIR   = '/content/outputs/classifications'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Загрузка метк ImageNet
import urllib.request
url = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'
urllib.request.urlretrieve(url, '/content/imagenet_labels.json')
with open('/content/imagenet_labels.json') as f:
    LABELS = json.load(f)

# ResNet50 претренированный
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(device).eval()
print(f'ResNet50 загружен, устройство: {device}')

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
results = []

with torch.no_grad():
    for fname in tqdm(frames, desc='ResNet'):
        img = Image.open(os.path.join(FRAMES_DIR, fname)).convert('RGB')
        inp = transform(img).unsqueeze(0).to(device)
        out = model(inp)
        probs = torch.softmax(out, dim=1)[0]
        top5 = torch.topk(probs, 5)
        for rank, (idx, prob) in enumerate(zip(top5.indices.tolist(), top5.values.tolist()), 1):
            results.append({
                'frame': fname,
                'rank': rank,
                'class_id': idx,
                'class_name': LABELS[idx],
                'probability': round(prob, 4),
            })

df = pd.DataFrame(results)
df.to_csv(f'{OUTPUT_DIR}/resnet_classifications.csv', index=False)
print(f'Кадров обработано: {len(frames)}')
df[df.rank == 1].head(10)